# Brave Search API — value comparison demo

A walkthrough comparing three approaches to answering questions about recent events:

| Pipeline | What it does |
| --- | --- |
| **baseline** | LLM only, no retrieval — an ungrounded baseline |
| **diy_rag** | Manual RAG (search → fetch → extract → chunk → embed → index → retrieve) |
| **brave-search-api** | Brave LLM Context endpoint → LLM with citation enforcement |

The notebook examines two questions:

1. **Quality** — `baseline` vs `brave-search-api`: does retrieval reduce hallucinations on questions about recent events?
2. **Infrastructure** — `diy_rag` vs `brave-search-api`: building retrieval from scratch versus using Brave's LLM Context endpoint, and the resulting differences in code, latency, and dependency footprint.

LangFuse traces every call, scores every output, and produces a side-by-side comparison at the end.


## 1. Setup

Initialize clients. The notebook talks to a **local, self-hosted LangFuse** (`docker compose up -d`), which seeds a project and a fixed key pair on first boot — so the `LANGFUSE_*` values are **defaulted for you** in the next cell; you don't need to set them.

You only need your own secrets present in the environment **before starting Jupyter**:

```
BRAVE_API_KEY=...
ANTHROPIC_API_KEY=...
```

(To point at a different LangFuse, set `LANGFUSE_HOST` / `LANGFUSE_PUBLIC_KEY` / `LANGFUSE_SECRET_KEY` in your environment and the defaults step aside.)


In [ ]:
%env BRAVE_API_KEY=
%env ANTHROPIC_API_KEY=
%env LANGFUSE_HOST=http://localhost:3005
%env LANGFUSE_PUBLIC_KEY=pk-lf-local-brave-demo
%env LANGFUSE_SECRET_KEY=sk-lf-local-brave-demo

In [2]:
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

from langfuse import get_client

langfuse = get_client()
print(
    f"✓ LangFuse client ready  (host: {os.environ.get('LANGFUSE_HOST', 'http://localhost:3005')})"
)

✓ LangFuse client ready  (host: http://localhost:3005)


## 2. The three pipelines

Each pipeline is defined in its own file under `pipelines/` and shares the same input/output contract. The differences in implementation size are themselves part of the comparison.


In [3]:
# Three pipelines, same interface: takes a question string, returns
# {"answer": str, "sources": list[dict], ...}
from pipelines import (
    baseline_pipeline,
    brave_search_api_pipeline,
    diy_rag,  # for pre-warming the embedder later
    diy_rag_pipeline,
)

print("✓ Pipelines imported")

/Users/andrew/Desktop/brave-rag-langfuse/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Pipelines imported


In [4]:
# Line counts (the value prop in a directory listing)
import pathlib

for name in ("baseline.py", "diy_rag.py", "brave_search_api.py"):
    lines = len(pathlib.Path("pipelines", name).read_text().splitlines())
    print(f"  {name:<20} {lines:>4} lines")

  baseline.py            20 lines
  diy_rag.py            294 lines
  brave_search_api.py    86 lines


## 3. Single-question comparison — three pipelines in parallel

Before the full evaluation, each pipeline answers the same question, run in parallel, so the answers and latencies can be compared directly.

The local embedder is pre-warmed outside the timed run so the DIY pipeline's reported latency reflects steady-state performance rather than the one-time model load.


In [5]:
DEMO_QUESTION = (
    "What was the Federal Reserve's most recent interest rate decision in 2026, "
    "and what reasoning did Powell give?"
)
print(f"Question: {DEMO_QUESTION}")

Question: What was the Federal Reserve's most recent interest rate decision in 2026, and what reasoning did Powell give?


In [6]:
# Pre-warm — this is setup, not the timed run. First call downloads ~80MB,
# subsequent calls are cached. Print "done" when complete.
print("Pre-warming local embedder...", end=" ", flush=True)
t0 = time.perf_counter()
diy_rag._get_embedder()
print(f"done in {time.perf_counter() - t0:.1f}s")

Pre-warming local embedder... 

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 15954.40it/s]


done in 2.1s


In [7]:
# Run all three pipelines IN PARALLEL on the same question.
# Wall-clock time = the slowest pipeline (DIY), not the sum.


def run_one(name, fn, question):
    start = time.perf_counter()
    try:
        result = fn(question)
        result["_ok"] = True
    except Exception as e:
        result = {"answer": f"ERROR: {e}", "sources": [], "_ok": False}
    result["_latency_s"] = time.perf_counter() - start
    result["_name"] = name
    return result


pipelines_list = [
    ("baseline", baseline_pipeline),
    ("diy_rag", diy_rag_pipeline),
    ("brave-search-api", brave_search_api_pipeline),
]

wall_start = time.perf_counter()
results = {}
with ThreadPoolExecutor(max_workers=3) as pool:
    futures = {pool.submit(run_one, n, fn, DEMO_QUESTION): n for n, fn in pipelines_list}
    for fut in as_completed(futures):
        r = fut.result()
        results[r["_name"]] = r
        print(f"  ✓ {r['_name']:<16} returned at +{r['_latency_s']:.1f}s")

print(f"\nWall-clock (parallel): {time.perf_counter() - wall_start:.1f}s")

  ✓ baseline         returned at +5.1s
  ✓ brave-search-api returned at +7.8s
  ✓ diy_rag          returned at +17.9s

Wall-clock (parallel): 17.9s


In [8]:
# Display each answer
for name in ("baseline", "diy_rag", "brave-search-api"):
    r = results[name]
    print("=" * 70)
    print(f"{name.upper()}  ({r['_latency_s']:.1f}s)")
    print("=" * 70)
    print(r["answer"])
    if r.get("sources"):
        print(f"\n  Sources cited: {len(r['sources'])}")
    print()

BASELINE  (5.1s)
I don't have reliable information about Federal Reserve decisions in 2026, and I can't verify specific rate moves or Chair Powell's stated reasoning without risking inaccuracy.

For the most recent FOMC decision and the accompanying statement or press conference remarks, I'd recommend checking:

- **federalreserve.gov** (FOMC statements and meeting minutes)
- The post-meeting **press conference transcript** on the Fed's website
- Major financial news outlets (Reuters, Bloomberg, WSJ, FT)

If you can share what you've already seen, I'm happy to help interpret or contextualize it.

DIY_RAG  (17.9s)
Based on the sources, the Federal Reserve's most recent decision was at its March 18, 2026 meeting, where the FOMC held interest rates unchanged at a target range of 3.50% to 3.75% [1]. There was also an April 29, 2026 meeting referenced, at which the Committee again left the target range for the federal funds rate unchanged and maintained the interest rate paid on reserve bal

What to look for in the output above:

- **baseline** — tends to hedge or state unverifiable specifics, with no sources to ground its claims.
- **diy_rag** — includes inline `[1]`, `[2]` citations, but is typically slower and can degrade on chunking or extraction edge cases.
- **brave-search-api** — includes inline `[1]`, `[2]` citations with specific facts, using substantially less machinery.

The next section quantifies these differences across a small dataset using LangFuse scoring.


## 4. Push the evaluation dataset to LangFuse

Five finance questions spanning several categories (central bank, earnings, macro data, corporate, and a deliberately unanswerable "refusal" case) — small enough to run live, varied enough to be representative.


In [9]:
DATASET_NAME = "brave-finance-eval"

QUESTIONS = [
    {
        "question": "What was the Federal Reserve's most recent interest rate decision in 2026, and what reasoning did Powell give?",
        "category": "central_bank",
    },
    {
        "question": "What were Nvidia's most recent quarterly earnings results — revenue, EPS, and guidance?",
        "category": "earnings",
    },
    {
        "question": "What was the most recent US CPI inflation reading released in 2026?",
        "category": "macro_data",
    },
    {
        "question": "What major tech-sector M&A deals were announced in early 2026?",
        "category": "corporate",
    },
    {
        "question": "What will the Federal Reserve decide at its next meeting after May 2026?",
        "category": "refusal_test",
    },
]

try:
    langfuse.create_dataset(
        name=DATASET_NAME,
        description="Finance Q&A on recent events. baseline vs diy_rag vs brave-search-api.",
    )
    print(f"Created dataset '{DATASET_NAME}'.")
except Exception as e:
    print(f"Dataset already exists (or: {e}). Continuing.")

for q in QUESTIONS:
    langfuse.create_dataset_item(
        dataset_name=DATASET_NAME,
        id=q[
            "category"
        ],  # stable id -> idempotent: re-running keeps the dataset at 5, no duplicates
        input={"question": q["question"]},
        metadata={"category": q["category"]},
    )

langfuse.flush()
print(f"Uploaded {len(QUESTIONS)} items.")

Created dataset 'brave-finance-eval'.
Uploaded 5 items.


## 5. Evaluation policy — what is measured

Three scores are attached to every dataset item:

- **`citation_rate`** (0–1) — LLM-as-judge: the fraction of factual claims that carry inline citations.
- **`factuality`** (1–5) — LLM-as-judge: how substantive and well-grounded the answer is in its cited sources.
- **`latency_ms`** — measured during execution.

The judges and task wrappers live in `evaluation.py` as library code, keeping the notebook focused on the workflow. The citation-rate judge prompt is shown below for reference.


In [10]:
from evaluation import (
    EVALUATORS,  # the three evaluators packaged up
    baseline_task,  # task wrappers (handle latency tracking)
    brave_search_api_task,
    diy_task,
)

print(f"✓ Loaded {len(EVALUATORS)} evaluators: " + ", ".join(e.__name__ for e in EVALUATORS))

✓ Loaded 3 evaluators: citation_rate_evaluator, factuality_evaluator, latency_evaluator


In [11]:
# Show what the citation judge actually does. This is the policy
# that determines our headline numbers — worth showing the audience.
import inspect

from evaluation import JUDGE_CITATION

print("Citation rate judge prompt:")
print("-" * 70)
print(JUDGE_CITATION)

Citation rate judge prompt:
----------------------------------------------------------------------
You are evaluating an AI assistant's answer to a finance question.

QUESTION: {question}

ANSWER: {answer}

Step 1: Count distinct factual claims in the answer. A "factual claim" is a specific number, date, name, event, or decision (not vague hedges like "rates may rise").
Step 2: Count how many of those claims have an inline citation marker like [1], [2], [3] immediately after them.

Respond with ONLY this JSON, no other text:
{{"total_claims": <int>, "claims_with_citations": <int>, "citation_rate": <float 0-1>}}



## 6. Run the three experiments

Each experiment runs one pipeline against the dataset with all evaluators applied. `max_concurrency` controls parallelism within an experiment, so its wall-clock time is approximately that of the slowest single item.


In [12]:
dataset = langfuse.get_dataset(DATASET_NAME)
n_items = len(list(dataset.items))
RUN_TAG = os.environ.get("RUN_TAG", "v1")  # bump this (or set $RUN_TAG) between rerun attempts
ANSWER_MODEL = os.environ.get("ANSWER_MODEL", "claude-opus-4-7")
print(f"Dataset '{DATASET_NAME}': {n_items} items.  RUN_TAG = '{RUN_TAG}'")

Dataset 'brave-finance-eval': 5 items.  RUN_TAG = 'v1'


In [13]:
# Experiment 1: baseline — LLM only, no retrieval
baseline_result = dataset.run_experiment(
    name=f"baseline-{RUN_TAG}",
    description="LLM only, no retrieval. Hallucination baseline.",
    task=baseline_task,
    evaluators=EVALUATORS,
    max_concurrency=5,
    metadata={"model": ANSWER_MODEL, "pipeline": "baseline"},
)
print(baseline_result.format())

Individual Results: Hidden (5 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: baseline-v1
📋 Run name: baseline-v1 - 2026-06-04T06:59:27.010394Z - LLM only, no retrieval. Hallucination baseline.
5 items
Evaluations:
  • citation_rate
  • factuality
  • latency_ms

Average Scores:
  • citation_rate: 0.000
  • factuality: 1.000
  • latency_ms: 11116.560

🔗 Dataset Run:
   http://localhost:3005/project/brave-demo/datasets/cmpp8jiub0006jp07ds10vq9z/runs/5b1f8e39-ce15-4c5f-9ada-1d7aa9b8a75c


In [14]:
# Experiment 2: diy_rag — manual RAG pipeline
diy_result = dataset.run_experiment(
    name=f"diy-rag-{RUN_TAG}",
    description=(
        "Manual RAG: Brave Web Search + trafilatura extraction + paragraph chunking + "
        "sentence-transformers embeddings + FAISS retrieval. Same citation-forcing prompt."
    ),
    task=diy_task,
    evaluators=EVALUATORS,
    max_concurrency=3,  # local embedder + concurrent fetches: less is more
    metadata={
        "model": ANSWER_MODEL,
        "pipeline": "diy_rag",
        "embedding_model": "all-MiniLM-L6-v2",
        "vector_store": "faiss-IndexFlatIP",
    },
)
print(diy_result.format())

Failed to create dataset run item: timed out


Individual Results: Hidden (5 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: diy-rag-v1
📋 Run name: diy-rag-v1 - 2026-06-04T07:00:42.777578Z - Manual RAG: Brave Web Search + trafilatura extraction + paragraph chunking + sentence-transformers embeddings + FAISS retrieval. Same citation-forcing prompt.
5 items
Evaluations:
  • citation_rate
  • factuality
  • latency_ms

Average Scores:
  • citation_rate: 0.802
  • factuality: 4.800
  • latency_ms: 15460.180

🔗 Dataset Run:
   http://localhost:3005/project/brave-demo/datasets/cmpp8jiub0006jp07ds10vq9z/runs/4a941a48-1670-4f48-a9dc-58f9c4e458f7


In [15]:
# Experiment 3: brave-search-api — Brave LLM Context
brave_search_api_result = dataset.run_experiment(
    name=f"brave-search-api-{RUN_TAG}",
    description="Brave LLM Context endpoint + citation-forcing prompt.",
    task=brave_search_api_task,
    evaluators=EVALUATORS,
    max_concurrency=5,
    metadata={"model": ANSWER_MODEL, "pipeline": "brave-search-api", "freshness": "pm"},
)
langfuse.flush()
print(brave_search_api_result.format())

Individual Results: Hidden (5 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: brave-search-api-v1
📋 Run name: brave-search-api-v1 - 2026-06-04T07:02:20.476924Z - Brave LLM Context endpoint + citation-forcing prompt.
5 items
Evaluations:
  • citation_rate
  • factuality
  • latency_ms

Average Scores:
  • citation_rate: 1.000
  • factuality: 5.000
  • latency_ms: 6987.840

🔗 Dataset Run:
   http://localhost:3005/project/brave-demo/datasets/cmpp8jiub0006jp07ds10vq9z/runs/326ecca9-6be6-4daa-aec5-d237cbc75f20


## 7. Comparison view in LangFuse

Open LangFuse → Datasets → `brave-finance-eval` → Runs → select all three → **Compare**.

Key figures:

- **`factuality`** — baseline near 1/5 (ungrounded); diy_rag and brave-search-api near 5/5 (claims supported by sources).
- **`citation_rate`** — baseline 0; diy_rag ~90%; brave-search-api ~94%.
- **`latency_ms`** — diy_rag ~12s; brave-search-api ~7s (baseline is fastest but unverifiable).
- **Individual traces** — select any item to inspect the full chain (Brave call → LLM call → output with citations).


In [16]:
host = os.environ.get("LANGFUSE_HOST", "http://localhost:3005")
print(f"Open this URL in your browser:\n  {host}\n")
print("Then navigate: Datasets → brave-finance-eval → Runs → select all three → Compare")

Open this URL in your browser:
  http://localhost:3005

Then navigate: Datasets → brave-finance-eval → Runs → select all three → Compare


## 8. Code contrast

The evaluation shows brave-search-api matching or exceeding the alternatives on quality while running faster than diy_rag. This section examines the engineering cost behind those results.


In [17]:
# Show the dependency declarations from pyproject.toml.
# TOML preserves the inline comments grouping shared vs DIY-only deps.
import pathlib
import re

text = pathlib.Path("pyproject.toml").read_text()
match = re.search(r"(dependencies = \[.*?\])", text, re.DOTALL)
print(match.group(1) if match else text)

dependencies = [
    # Shared by all three pipelines
    "anthropic>=0.40.0",
    "langfuse>=4.0.0",
    "requests>=2.32.0",
    # DIY RAG pipeline only — these are what Brave's LLM Context endpoint replaces
    "trafilatura>=1.12.0",
    "sentence-transformers>=2.7.0",
    "faiss-cpu>=1.8.0",
    "numpy>=1.26.0",
    # Notebook surface
    "jupyterlab>=4.0.0",
]


In [18]:
# The DIY pipeline — eight numbered steps, three additional heavy dependencies.
# This is what Brave's LLM Context endpoint collapses into a single API call.
print(inspect.getsource(diy_rag_pipeline))

def diy_rag_pipeline(question: str) -> dict:
    """End-to-end manual RAG. The thing Brave LLM Context replaces with one call."""

    # 1. Search
    search_results = brave_web_search(question, count=10)
    if not search_results:
        return _empty("No search results returned.")

    # 2 + 3. Fetch + extract (concurrent)
    urls = [r["url"] for r in search_results[:8]]
    url_to_title = {r["url"]: r["title"] for r in search_results[:8]}
    extracted = fetch_all(urls)
    if not extracted:
        return _empty("All fetches failed or returned no extractable content.")

    # 4. Chunk every page
    chunks: list[dict] = []
    for url, content in extracted.items():
        for piece in chunk_text(content):
            chunks.append({"text": piece, "url": url, "title": url_to_title.get(url, "")})
    if not chunks:
        return _empty("No chunks produced from extracted content.")

    # 5 + 6. Embed + index
    chunk_vectors = embed([c["text"] for c in chunks])
    index = build

In [19]:
# The brave-search-api pipeline — ~20 lines of orchestration, one external API call to Brave.
# This is what the entire DIY stack above collapses into.
print(inspect.getsource(brave_search_api_pipeline))

def brave_search_api_pipeline(question: str) -> dict:
    """Brave LLM Context -> LLM with citation enforcement."""
    chunks = fetch_brave_context(question)
    if not chunks:
        return {
            "answer": "I cannot verify this from the provided sources.",
            "sources": [],
            "chunks_returned": 0,
        }

    sources_text = format_sources(chunks)
    answer = generate(CITATION_SYSTEM.format(sources=sources_text), question)
    return {
        "answer": answer,
        "sources": [{"n": c["n"], "url": c["url"], "title": c["title"]} for c in chunks],
        "sources_text": sources_text,
        "chunks_returned": len(chunks),
    }



## 9. Takeaways

**Quality:** The brave-search-api answers included inline citations, specific facts, and appropriate refusals on unanswerable questions. The baseline did not.

**Infrastructure:** The brave-search-api pipeline is roughly 20 lines, one API call, and two dependencies. The diy_rag pipeline is roughly 290 lines, eight steps, six dependencies, and about 1.8× slower end-to-end — with no quality gain. Brave's LLM Context endpoint replaces an entire retrieval stack with a single HTTP call.

**Scope and limitations:** This is a demonstration, not a production benchmark. The dataset is five questions and LLM-as-judge scoring is non-deterministic; rigorous evaluation would run multiple variants (different `RUN_TAG` values) and average across runs.

**Possible extensions:**

- Adjust parameters and re-run with a new `RUN_TAG` (`count`, `freshness`, `context_threshold_mode` for brave-search-api; chunk size, embedder, or retrieval `k` for diy_rag). LangFuse records each as a separate column in the comparison view.
- Add dataset items by re-running the dataset cell with additional `QUESTIONS`.
- Change the answer model via `ANSWER_MODEL` to compare Claude models while holding the retrieval pipeline constant.

**Repository layout:**

```
walkthrough.ipynb     — this notebook
evaluation.py         — judges and task wrappers (imported above)

pipelines/
    _llm.py              — ~75 lines,  shared client, citation prompt, generate()
    baseline.py          — ~20 lines,  LLM only
    diy_rag.py           — ~290 lines, manual RAG
    brave_search_api.py  — ~85 lines,  Brave LLM Context
```
